# Blue Archive Gacha Simulation

In [ ]:
from dataclasses import dataclass, field
from enum import Enum, auto
import random
import statistics
from typing import Callable
import matplotlib.pyplot as plt

class PullType(Enum):
    STAR1=auto(); STAR2=auto(); STAR3_OTHER=auto(); STAR3_OFF=auto(); STAR3_PU=auto()

@dataclass
class GachaConfig:
    pickup_rate: float=0.007
    offpickup_rate: float=0.003
    star3_rate: float=0.03
    use_new_system: bool=True

@dataclass
class PullResult:
    pull_index:int
    pull_type:PullType
    by_charge:bool=False

@dataclass
class SimulationState:
    pull_count:int=0
    charge:int=0
    pickup_count:int=0
    offpickup_count:int=0
    star3_count:int=0
    history:list=field(default_factory=list)


In [ ]:
class GachaEngine:
    def __init__(self,cfg,rng=None):
        self.cfg=cfg
        self.rng=rng or random.Random()

    def _record(self,state,res):
        state.pull_count+=1
        state.charge+=1
        if res.pull_type in (PullType.STAR3_OTHER,PullType.STAR3_OFF,PullType.STAR3_PU):
            state.star3_count+=1
        if res.pull_type==PullType.STAR3_OFF:
            state.offpickup_count+=1
        if res.pull_type==PullType.STAR3_PU:
            state.pickup_count+=1
            if self.cfg.use_new_system:
                state.charge=0
        state.history.append(res)

    def pull(self,state):
        if self.cfg.use_new_system:
            if state.charge==199:
                res=PullResult(state.pull_count+1,PullType.STAR3_PU,True)
                self._record(state,res); return res
            if state.charge==99:
                pt=PullType.STAR3_PU if self.rng.random()<0.5 else PullType.STAR3_OFF
                res=PullResult(state.pull_count+1,pt,True)
                self._record(state,res); return res
        r=self.rng.random()
        if r<self.cfg.pickup_rate: pt=PullType.STAR3_PU
        elif r<self.cfg.pickup_rate+self.cfg.offpickup_rate: pt=PullType.STAR3_OFF
        elif r<self.cfg.star3_rate: pt=PullType.STAR3_OTHER
        elif r<0.215: pt=PullType.STAR2
        else: pt=PullType.STAR1
        res=PullResult(state.pull_count+1,pt,False)
        self._record(state,res); return res


In [ ]:
def stop_at_200(state): return state.pull_count>=200
def stop_on_pickup(state): return state.pickup_count>=1
def stop_pickup_or_200(state): return state.pickup_count>=1 or state.pull_count>=200

def simulate_once(engine,stop):
    s=SimulationState()
    while not stop(s):
        engine.pull(s)
    return s

def simulate(cfg,n_sim,stop,seed=0):
    rng=random.Random(seed)
    eng=GachaEngine(cfg,rng)
    return [simulate_once(eng,stop) for _ in range(n_sim)]


In [ ]:
def summarize(results):
    pulls=[r.pull_count for r in results]
    stars=[r.star3_count for r in results]
    pickup=[r.pickup_count for r in results]
    print("PU入手率",sum(x>0 for x in pickup)/len(results))
    print("平均★3",statistics.mean(stars))
    print("平均連数",statistics.mean(pulls))
    print("中央値",statistics.median(pulls))
    print("90%",statistics.quantiles(pulls,n=10)[8])
    print("95%",statistics.quantiles(pulls,n=20)[18])
    print("★3ゼロ",sum(x==0 for x in stars)/len(results))
    plt.hist(pulls,bins=30)
    plt.xlabel("Pulls"); plt.ylabel("Count")
    plt.show()


In [ ]:
cfg=GachaConfig(
    pickup_rate=0.007,
    offpickup_rate=0.003,
    star3_rate=0.03,
    use_new_system=True
)
results=simulate(cfg,10000,stop_pickup_or_200,seed=42)
summarize(results)
